# SITCOM-2104 - https://rubinobs.atlassian.net/browse/SITCOM-2104

Data were collected to establish a baseline for the VMS data and the FCUs on May 18, 2025 between 15h and 16h CLT (BLOCK-T499). During that time, the telescope was not tracking. The crew collected data at different elevation angles: 20º, 40º, 60º, and 80º and at different FCU speed: 0, 500, 1000, 1500, 2000 and 2550 RPM

The notebook is using data from the M1M3 Vibration Monitor System (VMS). These data needs to be copied at USDF from the vms-data server at the summit.
Some files are available in /scratch/users/b/boutigny/vmsdata, they have been converted in parquet format for convenience.

**Pre-requisite**

This notebook needs the **DateTimeRange** library. It is very a convenient library to handle time ranges and to compute overlaps between them

- pip install DateTimeRange

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

# %load_ext lab_black

import pandas as pd
import numpy as np

from astropy.time import Time
from astropy.time import TimeDelta
from datetime import datetime, timedelta

import matplotlib.pyplot as plt
from matplotlib import ticker
from matplotlib.dates import DateFormatter
import matplotlib

from lsst_efd_client import EfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState, TMAEvent
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient, getSubTopics
from lsst.ts.xml.enums import MTM1M3, MTMount

from scipy.signal import stft
from scipy import signal
from scipy.fft import fft, rfft, fftfreq, rfftfreq

from datetimerange import DateTimeRange

In [ ]:
async def get_time_range_by_axis(client, TMA_axis, begin, end):
    """
    Query the status of the TMA on a given axis (elevation or azimuth) during a time window and return
    a list of time ranges where the TMA  is still on this axis.

    Args:
        client: EFD client

        TMA_axis (str): Axis of the TMA to be checked ("elevation" or "azimuth").

        begin (string) : Starting time

        end (string) : End time

    Returns:
        time_range (list of DateTimeRange) : Range of times where the TMA is still on the considered TMA_axis
    """

    topic = f"lsst.sal.MTMount.logevent_{TMA_axis}MotionState"
    df_status = getEfdData(client, topic, begin=Time(begin), end=Time(end))
    if len(df_status) == 0:
        print(f"No status data found for {TMA_axis} axis, using full time range")
        time_ranges = [
            DateTimeRange(
                pd.Timestamp(begin.datetime64, tz="UTC"),
                pd.Timestamp(end.datetime64, tz="UTC"),
            )
        ]
        return time_ranges

    select_stopped = df_status["state"] == 1

    # Select rows corresponding to motion state == stopped
    i_start = np.array(
        [
            df_status.index.get_loc(df_status[select_stopped].index[i])
            for i in range(len(df_status[select_stopped]))
        ]
    )
    t_start = list(df_status["state"].index[i_start])
    if i_start[-1] < len(df_status) - 1:
        t_end = list(df_status["state"].index[i_start + 1])
    else:
        t_end = list(df_status["state"].index[i_start[0:-1] + 1])
        t_end.append(pd.Timestamp(end, tz="UTC"))
    time_ranges = []
    for i, (t1, t2) in enumerate(zip(t_start, t_end)):
        time_ranges.append(DateTimeRange(t1, t2))

    return time_ranges


def get_overlaps(t_range_1, t_range_2, min_delta):
    """
    Find the time overlaps between two lists of DateTimeRange. The overlap will only be considered if its duration is
    > min_delta seconds

    Args:
        t_range_1 (list of DateTimeRange)
        t_range_2 (list of DateTimeRange)
        min_delta (int) minimum duration in seconds of the time overlap

    Return:
        overlaps (list of DateTimeRange)
    """
    overlaps = []
    for r1 in t_range_1:
        for r2 in t_range_2:
            t_range_ok = r1.intersection(r2)
            if (t_range_ok.start_datetime != None) and (
                t_range_ok.timedelta.total_seconds() > min_delta
            ):
                overlaps.append(
                    DateTimeRange(t_range_ok.start_datetime, t_range_ok.end_datetime)
                )

    return overlaps

In [ ]:
def get_psd(vals, timestep, min_freq=0):
    """
    Compute all PSD related quantities

    Args:
        vals (array of floats): array containing the signal to be analyzed
        timestep (float): time interval in seconds beween 2 samples
        min_freq (float): minimum frequencyin Hz to be used to compute the integrated rms displacement. The displacement will be integrated
        between min_freq and 100 Hz

    Return:
        tuple:
        freqs_sel(array of floats): output frequencies in Hz from the Fourier analysis
        psd_accel_sel (array of float): the PSD acceleration in the selected frequency interval
        rms_accel (array of float): RMS of the acceleration in m/s²
        rms_disp (array of floats): RMS of the displacement
        cum_disp (array of floats) : cumulative RMS of the displacement - The sum is computed from the high frequencies to the lower ones
        total_disp (float) : integrated displacement in meter. This is the total displacement computer within the considered frequency interval
    """

    # Remove the mean from the signal.
    meanval = np.mean(vals)
    signal = vals - meanval
    fs = 1 / timestep

    N_samples = len(signal)

    # Compute real FFT of acceleration signal
    X_a = np.fft.rfft(signal)

    # compute frequency bins
    freqs = np.fft.rfftfreq(N_samples, timestep)

    # Compute PSD of acceleration
    psd_accel = (np.abs(X_a) ** 2) / (N_samples * fs)

    # Cut low frequencies - We should at least cut the 0 frequency to avoid a division by 0
    sel = freqs > min_freq
    freqs_sel = freqs[sel]
    psd_accel_sel = psd_accel[sel]

    # Compute PSD of displacement
    psd_disp = psd_accel_sel / (2 * np.pi * freqs_sel) ** 4

    # Compute the RMS acceleration and displacement
    df = freqs_sel[1] - freqs_sel[0]
    rms_accel = np.sqrt(psd_accel_sel * df)
    rms_disp = np.sqrt(psd_disp * df)

    # Compute cumulative displacement and total displacement
    # We accumulate the displacement starting from the higher frequencies toward the lower frequuencies
    # in order to be able to see the jumps corresponding to peaks in frequencies
    cum_disp = np.sqrt(np.cumsum(psd_disp[::-1] * df))[::-1]
    total_disp = cum_disp[0]

    return (freqs_sel, psd_accel_sel, rms_accel, rms_disp, cum_disp, total_disp)

In [ ]:
def plot_spectrogram(
    vms_data,
    vms_date,
    fan_speed,
    f_min=0,
    f_max=100,
    nperseg=4096,
    psd_min=1.0e-8,
    psd_max=1.0e-6,
    delta_t=30,
    savefig=False,
):
    """
    Plot the spectrograms corresponding to the vms_data data sample for the 3 axes of the 3 VMS accelerometers

    Args:
        vms_data (dataframe): data sample
        vms_date (string): data acquisition date
        fan_speed (int): fan speed in RPM
        f_min (int): minimum frequency
        f_max (int): maximumn frequency
        nperseg (int): width of the window for the FFT (should be a power of 2)
        psd_min (float): minimum value of the psd of acceleration to be used in the color scale
        psd_max (float): maximum value of the psd of acceleration to be used in the color scale
        delta_t (float): time in seconds at which the change of speed of the fans occur. A vertical dotted line will be displayed.
        savefig (boolean): save plot in png format if True

    Return:
        None
    """
    fig, ax = plt.subplots(3, 3, dpi=128, figsize=(10, 7))
    ax2 = ax

    for c in range(3):
        fs = 1 / np.mean(np.diff(Time(vms_data["times"]).unix))
        for j, axis in enumerate("xyz"):
            key = f"m1m3_{axis}_{c + 1}"

            f, t, Zxx = stft(vms_data[key], fs, "hamming", nperseg, scaling="psd")

            pcm = ax[c][j].pcolormesh(
                t,
                f,
                np.abs(Zxx),
                shading="gouraud",
                norm=matplotlib.colors.LogNorm(vmin=psd_min, vmax=psd_max),
            )
            ax[c][j].set(
                xlabel="Time [s]",
                ylabel="Frequency [Hz]",
                title=f"Sensor {c + 1} - axis {axis}",
            )
            ax[c][j].set_ylim([f_min, f_max])
            ax[c][j].axvline(delta_t, c="black", ls="dotted", lw=1)
            fig.colorbar(pcm, ax=ax[c][j])
    fig.suptitle(f"{vms_date} - Fan speed = {fan_speed} RPM")
    fig.tight_layout()
    if savefig:
        fig.savefig(f"spectrogram_{vms_date}_{fan_speed}_RPM_{f_min}-{f_max}.png")

    return

In [ ]:
client = EfdClient("usdf_efd")

In [ ]:
# This is roughly the period of time during which we expect to find the data corresponding to BLOCK-T499
selected_date = "2025-05-18"
t_start = Time(f"{selected_date} 19:00:00")
t_end = Time(f"{selected_date} 20:30:00")

df_mtm1m3ts_fan_demand = getEfdData(
    client, "lsst.sal.MTM1M3TS.command_heaterFanDemand", begin=t_start, end=t_end
)

In [ ]:
# Find which fan speeds are available and when
fan_speed_avail = list(set(df_mtm1m3ts_fan_demand["fanRPM0"]))
fan_speed_avail.sort()

# Initialize a dictionnary to store time periods where the FCUs were running at a given speed
fan_delta_t = {}
for speed in fan_speed_avail:
    cut = df_mtm1m3ts_fan_demand["fanRPM0"] == speed
    pos_true = np.where(cut)[0]
    fan_delta_t[speed] = []
    for p in pos_true:
        if p < len(df_mtm1m3ts_fan_demand) - 1:
            t1 = df_mtm1m3ts_fan_demand.index[p]
            t2 = df_mtm1m3ts_fan_demand.index[p + 1]
            fan_delta_t[speed].append([t1, t2])
        else:
            t1 = df_mtm1m3ts_fan_demand.index[p]
            t2 = t1 + pd.Timedelta(value=60, unit="seconds")
            fan_delta_t[speed].append([t1, t2])

In [ ]:
# We select periods where the FCUs are running at a given speed for at least "min_duration" seconds
min_duration = 30
selected_periods = {}

for fan_speed in fan_speed_avail:
    print(f"Fan speed: {fan_speed}")
    selected_periods[fan_speed] = []
    for inst in fan_delta_t[fan_speed]:
        time_range = DateTimeRange(inst[0], inst[1])
        duration = time_range.get_timedelta_second()
        if duration > min_duration and inst[0].date().isoformat() == selected_date:
            print(inst[0], f"{duration:.0f} s")
            selected_periods[fan_speed].append(time_range)

## Retrieve VMS M1M3 data corresponding to the selected date and load them into a pandas dataframe

In [ ]:
# Acquisition date of VMS data
vms_date = "2025-05-18"

vms_top_dir = "/scratch/users/b/boutigny/vmsdata"
year, month = vms_date.split("-")[0:2]

# Directory containing VMS data
vms_dir = os.path.join(vms_top_dir, year, month)

# Check if a parquet file exists
vms_m1m3_parquet_filename = os.path.join(vms_dir, "M1M3-" + vms_date + "T00:00.parquet")
if os.path.isfile(vms_m1m3_parquet_filename):
    print(f"Reading VMS data from parquet file:{vms_m1m3_parquet_filename}")
    vms_m1m3_data = pd.read_parquet(vms_m1m3_parquet_filename)
else:
    print("Parquet file not found")

# Try to avoid that corrections in the next cell to be applied more than 1 time
applied_corrections = False

## Correct VMS data to get the time in UTC and the acceleration in m/s²

In [ ]:
# Check that the corrections has not been applied already
if applied_corrections == False:
    # Subtract 37s to timestamps in order to take into account the difference between the TAI and UTC
    vms_m1m3_data.times = vms_m1m3_data.times - pd.Timedelta(value=37, unit="seconds")

    # Convert acceleration from milli-g to m/s²
    for sensor in range(3):
        for axis in "xyz":
            key = f"m1m3_{axis}_{sensor + 1}"
            vms_m1m3_data[key] = 1e-3 * 9.8 * vms_m1m3_data[key]

    applied_corrections = True
else:
    print("Corrections have already been applied - Skipped")

In [ ]:
# We define the time window in which we will get the data
t1 = f"{vms_date} 19:00:00"
t2 = f"{vms_date} 20:15:59"

# We get the list of time periods where the TMA is still on the azimuth or on the elevation axis
tr_az = await get_time_range_by_axis(client, "azimuth", t1, t2)
tr_el = await get_time_range_by_axis(client, "elevation", t1, t2)

min_delta = (
    50  #  We want a VMS dataset lasting at least this amount of time (in seconds)
)

# Get the list of time periods where the TMA is still on both axes
TMA_still = get_overlaps(tr_az, tr_el, min_delta)

In [ ]:
TMA_still

In [ ]:
# Find the dates which are the closest to the beginning and end of the selected data period in the mtm1m3ts_fan_demand dataframe index
t1_df_fan = df_mtm1m3ts_fan_demand.index.searchsorted(t1)
t2_df_fan = df_mtm1m3ts_fan_demand.index.searchsorted(t2)

In [ ]:
# RetrieveTMA elevation angles from the EFD
efd_begin = Time(df_mtm1m3ts_fan_demand.index[t1_df_fan:t2_df_fan][0].to_datetime64())
tmp_time = df_mtm1m3ts_fan_demand.index[t1_df_fan:t2_df_fan][-1]
if tmp_time < TMA_still[-1].end_datetime:
    tmp_time = TMA_still[-1].end_datetime
efd_end = Time(tmp_time.to_datetime64())
df_azi = getEfdData(
    client,
    "lsst.sal.MTMount.elevation",
    begin=efd_begin,
    end=efd_end,
)

In [ ]:
# Plot fan speed as a function of time and the periods where the TMA is still on both axes
# This will help to select interesting events

fig, ax = plt.subplots(1, 1, dpi=128, figsize=(12, 5))
times = df_mtm1m3ts_fan_demand.index[t1_df_fan:t2_df_fan]
times_ext = times.union([times[-1] + pd.Timedelta(value=600, unit="seconds")])
fs = df_mtm1m3ts_fan_demand["fanRPM0"][t1_df_fan:t2_df_fan] * 10
for i in range(len(times_ext) - 1):
    if i == 0:
        ax.plot(
            times_ext[i : i + 2], [fs.iloc[i], fs.iloc[i]], c="green", label="Fan speed"
        )
    else:
        ax.plot(times_ext[i : i + 2], [fs.iloc[i], fs.iloc[i]], c="green")
    ax.plot(times_ext[i : i + 2], [fs.iloc[i], fs.iloc[i]], c="green")
    if i < len(times_ext) - 2:
        ax.plot(
            [times_ext[i + 1], times_ext[i + 1]],
            [fs.iloc[i], fs.iloc[i + 1]],
            c="green",
        )

# Superimpose time periods where the TMA is still
for i, p in enumerate(TMA_still):
    if i == 0:
        ax.plot(
            [p.start_datetime, p.end_datetime], [-100, -100], c="red", label="TMA still"
        )
    else:
        ax.plot([p.start_datetime, p.end_datetime], [-100, -100], c="red")

ax2 = ax.twinx()
ax2.plot(df_azi.index, df_azi["actualPosition"], c="blue", label="Elevation")
ax2.set_ylim([0, 90])
ax2.set_ylabel("Elevation (degrees)")
ax2.legend()

ax.legend()
ax.set_xlabel("Time")
ax.set_ylabel("Fan speed in RPM")
ax.xaxis.set_major_formatter(DateFormatter("%H-%M-%S"))
plt.setp(ax.get_xticklabels(), rotation=45)
plt.grid()
fig.suptitle(f"FCU fan speeds - Date : {vms_date}")
fig.tight_layout()
fig.savefig(f"fan_speed_{vms_date}.png")

## Here we prepare the main plot which will show the M1M3 displacement as a function of the FCU speeds and for several elevation angles.

In [ ]:
# The following dictionary from the graph above.
data_sequence = {"elevation": [80, 80, 80, 80, 80, 80, 60, 60, 60, 60, 60, 60, 40, 40, 40, 40, 40, 40, 20, 20, 20, 20, 20, 20],
                 "fan_speed": [0, 500, 1000, 1500, 2000, 2550, 0, 500, 1000, 1500, 2000, 2550, 0, 500, 1000, 1500, 2000, 2550, 0, 500, 1000, 1500, 2000, 2550],
                 "instance":  [0, 0, 0, 0, 2, 0, 1, 1, 1, 1, 3, 1, 2, 2, 2, 2, 4, 2, 3, 3, 3, 3, 5, 3]}  # fmt: skip
tab_sequence = pd.DataFrame(data_sequence)

In [ ]:
# Loop over all chunks of data and compute the integrated RMS displacement
delta_t = timedelta(seconds=10)
min_delta = 50  # Data chunk should last at least this amount of seconds
max_t = timedelta(
    seconds=300
)  # We keep at most 300s of data. This is to limit the computing time in case we get a really long data frame
total_disp_list = {
    "x": [],
    "y": [],
    "z": [],
}  # This corresponds to the 3 accelerometer axis

for index, row in tab_sequence.iterrows():
    elev = row["elevation"]
    fan_speed = row["fan_speed"] / 10
    inst = row["instance"]
    # print(index, elev, fan_speed, inst)
    selected_data = get_overlaps(selected_periods[fan_speed], TMA_still, min_delta)

    i1_run = vms_m1m3_data.times.searchsorted(
        selected_data[inst].start_datetime.tz_localize(None)
    )
    i2_run = vms_m1m3_data.times.searchsorted(
        min(
            selected_data[inst].end_datetime.tz_localize(None),
            selected_data[inst].start_datetime.tz_localize(None) + max_t,
        )
    )
    subdat = vms_m1m3_data[i1_run:i2_run]

    sensor = 3  # We use 1 single accelerometer
    for axis in "xyz":
        key = f"m1m3_{axis}_{sensor}"
        freq, psd_accel, rms_accel, rms_disp, cml_disp, total_disp = get_psd(
            subdat[key], np.mean(np.diff(Time(subdat["times"]).unix)), min_freq=0.05
        )
        total_disp_list[axis].append(total_disp)

In [ ]:
# Store total displacement in pandas dataframe and convert to micrometers
for axis in "xyz":
    tab_sequence[f"total_disp_{axis}"] = np.array(total_disp_list[axis]) * 1.0e6

In [ ]:
# Plot results
avail_elevations = np.sort(np.array(list(set(tab_sequence["elevation"]))))
fig, ax = plt.subplots(3, 4, figsize=(15, 10))

for i, elev in enumerate(avail_elevations):
    subdat = tab_sequence[tab_sequence["elevation"] == elev]
    for j, axis in enumerate("xyz"):
        ax[j][i].plot(subdat["fan_speed"], subdat[f"total_disp_{axis}"], marker="o")
        ax[j][i].set_xlabel("Fan speed (RPM)")
        ax[j][i].set_ylabel("Total displacement (microns)")
        ax[j][i].set_title(f"Elev: {elev} degrees - Axis: {axis}")
        ax[j][i].set_ylim([0, 0.25])

fig.suptitle("BLOCK-T499 - FCU Vibration Analysis", fontsize=15)
fig.tight_layout()

## Conclusion

From the previous plot, we conclude that the total displacement computed from the M1M3 VMS stays well below 1 micron and does not show significant depenedence neither with the
speed of the FCUs nor with the elevation angle of the TMA.

## More detailed plots

The remaining cells in the notebook allow to create detailed plots corresponding to one specific event (FCU speed, elevation) 

In [ ]:
# We select the desired fan speed and the dataset number in case there is more than 1 occurence
fan_speed = 200  # Be careful here the fan speed is in units recorded in the EFD (this is RPM/10)
inst = 2

good_data = get_overlaps(selected_periods[fan_speed], TMA_still, min_delta)

# delta_t is the time period before the FCU change of speed that we want to keep in order to observe the effect
# of the change of speed on the frequencies of the vibrations.
delta_t = timedelta(seconds=10)
t11 = vms_m1m3_data.times.searchsorted(
    good_data[inst].start_datetime.tz_localize(None) - delta_t
)

# Select at most 300s of VMS data (we could use more but it is not necessary and it would increase the processing time)
t21 = vms_m1m3_data.times.searchsorted(
    min(
        good_data[inst].end_datetime.tz_localize(None),
        good_data[inst].start_datetime.tz_localize(None) + timedelta(seconds=300),
    )
)
vms_data = vms_m1m3_data[t11:t21]

In [ ]:
# Plot fan speed as a function of time and the periods where the TMA is still on both axes
# It also display the beginning of the chunk of data as a black vertical dotted line.
# This plot is intended to make sure that we have selected the right chunk of data

fig, ax = plt.subplots(1, 1, dpi=128, figsize=(12, 5))
times = df_mtm1m3ts_fan_demand.index[t1_df_fan:t2_df_fan]
times_ext = times.union([times[-1] + pd.Timedelta(value=600, unit="seconds")])
fs = df_mtm1m3ts_fan_demand["fanRPM0"][t1_df_fan:t2_df_fan] * 10
for i in range(len(times_ext) - 1):
    if i == 0:
        ax.plot(
            times_ext[i : i + 2], [fs.iloc[i], fs.iloc[i]], c="green", label="Fan speed"
        )
    else:
        ax.plot(times_ext[i : i + 2], [fs.iloc[i], fs.iloc[i]], c="green")
    ax.plot(times_ext[i : i + 2], [fs.iloc[i], fs.iloc[i]], c="green")
    if i < len(times_ext) - 2:
        ax.plot(
            [times_ext[i + 1], times_ext[i + 1]],
            [fs.iloc[i], fs.iloc[i + 1]],
            c="green",
        )

# Superimpose time periods where the TMA is still
for i, p in enumerate(TMA_still):
    if i == 0:
        ax.plot(
            [p.start_datetime, p.end_datetime], [-100, -100], c="red", label="TMA still"
        )
    else:
        ax.plot([p.start_datetime, p.end_datetime], [-100, -100], c="red")

# Draw a vertical line to indicate the start of the selected period
ax.axvline(
    good_data[inst].start_datetime.tz_localize(None), c="black", ls="dotted", lw=1
)

ax.set_xlim(
    [
        good_data[inst].start_datetime.tz_localize(None) - timedelta(seconds=300),
        good_data[inst].start_datetime.tz_localize(None) + timedelta(seconds=300),
    ]
)

ax.legend()
ax.set_xlabel("Time")
ax.set_ylabel("Fan speed in RPM")
ax.xaxis.set_major_formatter(DateFormatter("%H-%M-%S"))
plt.setp(ax.get_xticklabels(), rotation=45)
plt.grid()
fig.suptitle(f"FCU fan speeds - Date : {vms_date}")
fig.tight_layout()
fig.savefig(f"fan_speed_{vms_date}.png")

## Spectrograms

In [ ]:
# First in the full frequency range (0 - 100 Hz)
plot_spectrogram(
    vms_data,
    vms_m1m3_data["times"][t11],
    fan_speed * 10,
    f_min=0,
    f_max=100,
    nperseg=128,
    psd_min=1.0e-8,
    psd_max=1.0e-6,
    delta_t=delta_t.seconds,
    savefig=True,
)

In [ ]:
# and second in the low frequency part (< 40 Hz)
plot_spectrogram(
    vms_data,
    vms_m1m3_data["times"][t11],
    fan_speed * 10,
    f_min=0,
    f_max=40,
    nperseg=512,
    psd_min=1.0e-8,
    psd_max=1.0e-6,
    delta_t=delta_t.seconds,
    savefig=True,
)

## Comment on the previous plot

We can identity frequencies that are changing with time (see yellow lines for instance). The beginning of the frequency change is associated to the change of FCU speed (vertical black dotted line). This is clearly demonstrating that the VMS is able to detect vibration induced by the FCUs. We also see that the frequencies associated to the FCUs are relatively
high so the vibrations would have to be very strong to contribute significantly to the displacement.

# PSD analysis

In [ ]:
max_t = timedelta(seconds=300)  # We keep at most 300s of data

selected_data = get_overlaps(selected_periods[fan_speed], TMA_still, min_delta)

# get the indexes of the lines in the VMS array corresponding to the start and the end of the data chunk
i1_run = vms_m1m3_data.times.searchsorted(
    selected_data[inst].start_datetime.tz_localize(None)
)
i2_run = vms_m1m3_data.times.searchsorted(
    min(
        selected_data[inst].end_datetime.tz_localize(None),
        selected_data[inst].start_datetime.tz_localize(None) + max_t,
    )
)

## Plot the RMS of acceleration

In [ ]:
fig, ax = plt.subplots(3, 3, dpi=150, figsize=(8, 6))
subdat = vms_m1m3_data[i1_run:i2_run]

for c in range(3):
    for j, axis in enumerate("xyz"):
        key = f"m1m3_{axis}_{c + 1}"
        freq, psd_accel, rms_accel, rms_disp, cml_disp, total_disp = get_psd(
            subdat[key],
            np.mean(np.diff(Time(subdat["times"]).unix)),
            min_freq=0.1,
        )
        ax[c][j].plot(freq, rms_accel, zorder=9, lw=0.2)
        ax[c][j].set_xticks(np.arange(0, 110, 10))
        ax[c][j].set(
            ylabel="RMS Accel (m/s²)",
            xlabel="Frequency [Hz]",
            title=f"Sensor {c + 1} - axis {axis}",
        )
        yticks = ticker.MaxNLocator(4)
        ax[c][j].yaxis.set_major_locator(yticks)
        ax[c][j].set_yscale("log")
        plt.setp(ax[c][j].get_xticklabels(), rotation=45)
fig.suptitle(f"{vms_date} - Fan speed: {fan_speed * 10} RPM")
fig.tight_layout()

## Plot the RMS of displacement

In [ ]:
fig, ax = plt.subplots(3, 3, dpi=150, figsize=(8, 6))
subdat = vms_m1m3_data[i1_run:i2_run]

for c in range(3):
    for j, axis in enumerate("xyz"):
        key = f"m1m3_{axis}_{c + 1}"
        freq, psd_accel, rms_accel, rms_disp, cml_disp, total_disp = get_psd(
            subdat[key],
            np.mean(np.diff(Time(subdat["times"]).unix)),
            min_freq=0.1,
        )
        ax[c][j].plot(freq, rms_disp, zorder=9, lw=0.2)
        ax[c][j].set_xticks(np.arange(0, 110, 10))
        ax[c][j].set(
            ylabel="RMS disp (m)",
            xlabel="Frequency [Hz]",
            title=f"Sensor {c + 1} - axis {axis}",
        )
        yticks = ticker.MaxNLocator(4)
        ax[c][j].yaxis.set_major_locator(yticks)
        ax[c][j].set_yscale("log")
        plt.setp(ax[c][j].get_xticklabels(), rotation=45)
fig.suptitle(f"{vms_date} - Fan speed: {fan_speed * 10} RPM")
fig.tight_layout()

## Plot the integrated displacement

In [ ]:
fig, ax = plt.subplots(3, 3, dpi=150, figsize=(8, 6))
subdat = vms_m1m3_data[i1_run:i2_run]

for c in range(3):
    for j, axis in enumerate("xyz"):
        key = f"m1m3_{axis}_{c + 1}"
        freq, psd_accel, rms_accel, rms_disp, cml_disp, total_disp = get_psd(
            subdat[key],
            np.mean(np.diff(Time(subdat["times"]).unix)),
            min_freq=0.05,
        )
        ax[c][j].plot(
            freq,
            cml_disp * 1.0e6,
            label=rf"$\Sigma$: {total_disp * 1.0e6:.3f}$\mu m$",
            zorder=9,
            lw=1,
        )
        ax[c][j].set_xticks(np.arange(0, 110, 10))
        ax[c][j].set(
            ylabel=r"$\Sigma$disp ($\mu m$)",
            xlabel="Frequency [Hz]",
            title=f"Sensor {c + 1} - axis {axis}",
        )
        yticks = ticker.MaxNLocator(4)
        ax[c][j].set_ylim([1.0e-8, 1.0])
        ax[c][j].yaxis.set_major_locator(yticks)
        ax[c][j].set_yscale("log")
        ax[c][j].legend()
        plt.setp(ax[c][j].get_xticklabels(), rotation=45)
fig.suptitle(f"{vms_m1m3_data['times'][i1_run]} - Fan speed: {fan_speed * 10} RPM")
fig.tight_layout()

## Comment on the previous plot

The steps in the sub-plots correspond to the emerging frequencies that contribute to the displacement. They have a very small contribution to the total
displacement. Most of the displacement is associated to the first bins of the plots.